# 04 · Limpieza: Eventos Virtuales ACRIP 2026_anonimizada

Encuesta de satisfacción posterior a los eventos virtuales de ACRIP Región Central (ene 2025 – oct 2026). Una fila = una respuesta.
Misma estructura que el cuaderno 03; la única pregunta distinta es la **plataforma de conexión** (en vez de la locación).

- Entrada: `datos/originales/Eventos Virtuales ACRIP 2026_anonimizada.xlsx` (hoja `Responses`)
- Salida: `datos/limpios/04_eventos_virtuales_limpio.csv`

In [ ]:
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

## 1. Carga
Igual que en el 03: la fila 0 está vacía y los encabezados están en la fila 1, por eso `header=1`.

In [ ]:
df = pd.read_excel(ORIGINALES / "Eventos Virtuales ACRIP 2026_anonimizada.xlsx", sheet_name="Responses", header=1)
assert df.shape == (318, 19), df.shape
df.head()

## 2. Renombrar y eliminar columnas
Mismos nombres que el 03 para poder unir ambas bases. La calificación propia de lo virtual se llama `cal_plataforma`.
Se eliminan: `Unnamed: 0`, IP, teléfono y correo (vacías por la anonimización) y `Nombre del recopilador` (un solo valor, "…2025", aunque hay respuestas de 2026).

In [ ]:
COLUMNAS = {
    "ID_RESPUESTA": "id_respuesta",
    "Respuesta iniciada": "fecha_inicio",
    "Respuesta completada": "fecha_fin",
    "plataforma de conexión": "cal_plataforma",
    "convocatoria": "cal_convocatoria",
    "conferencistas": "cal_conferencistas",
    "temas tratados": "cal_temas",
    "temarios": "temas_sugeridos",
    "recibir actualizaciones": "recibir_info",
    "ID_PERSONA": "id_persona",
    "ID_EMPRESA": "id_empresa",
    "CIUDAD": "ciudad",
    "EMPRESA_EN_CRM": "empresa_en_crm",
    "METODO_ASIGNACION": "metodo_asignacion",
}
renombre = {col: nuevo for frag, nuevo in COLUMNAS.items() for col in df.columns if frag in col}
assert len(renombre) == len(COLUMNAS), renombre

df = df.rename(columns=renombre)[list(COLUMNAS.values())]
df.columns.tolist()

## 3. Fechas
Mismo formato en español que el 03 (`mar. 20, 2025 09:21:35 a. m.`), así que se usa la misma función `fecha_es`.

In [ ]:
MESES = {"ene": "01", "feb": "02", "mar": "03", "abr": "04", "may": "05", "jun": "06",
         "jul": "07", "ago": "08", "sep": "09", "sept": "09", "oct": "10", "nov": "11", "dic": "12"}


def fecha_es(serie):
    texto = (serie.str.replace("\xa0", " ")
                  .str.replace(r"^([a-z]+)\.", lambda m: MESES[m.group(1)], regex=True)
                  .str.replace("a. m.", "AM").str.replace("p. m.", "PM"))
    return pd.to_datetime(texto, format="%m %d, %Y %I:%M:%S %p")


df["fecha_inicio"] = fecha_es(df["fecha_inicio"])
df["fecha_fin"] = fecha_es(df["fecha_fin"])
df["fecha_evento"] = df["fecha_inicio"].dt.normalize()
df["anio"] = df["fecha_inicio"].dt.year
df["mes"] = df["fecha_inicio"].dt.month

assert df["fecha_inicio"].notna().all() and df["fecha_fin"].notna().all()
assert (df["fecha_fin"] >= df["fecha_inicio"]).all()
df["anio"].value_counts()

## 4. Ciudad
Aquí hay muchas más ciudades que en el 03 (es virtual, así que llega público de todo el país): 47 variantes para 36 ciudades.
Se normaliza a minúsculas y se mapea a la forma oficial con tilde. Dos casos dudosos:
- `Barra` → `Barranquilla` (única ciudad que empieza así entre las que aparecen).
- `Huy` → `Sin dato` (no corresponde a ninguna ciudad y la respuesta es anónima, no hay cómo verificarla).

Si aparece una variante nueva, el `assert` falla y obliga a agregarla. Vacíos → `Sin dato`.

In [ ]:
CIUDADES = {
    "bogotá": "Bogotá", "bogota": "Bogotá", "bogotá d.c.": "Bogotá", "bogotá, d.c.": "Bogotá",
    "bogotá d.c": "Bogotá", "bogota d.c.": "Bogotá", "bogota (bogota)": "Bogotá", "bogots": "Bogotá",
    "medellín": "Medellín", "medellin": "Medellín",
    "barranquilla": "Barranquilla", "barra": "Barranquilla",
    "cali": "Cali", "manizales": "Manizales", "cartagena": "Cartagena", "pereira": "Pereira",
    "villavicencio": "Villavicencio", "rionegro": "Rionegro", "ibagué": "Ibagué",
    "bucaramanga": "Bucaramanga", "guarne": "Guarne", "chia": "Chía", "guamal": "Guamal",
    "santa rosa de cabal": "Santa Rosa de Cabal", "facatativá": "Facatativá", "apartadó": "Apartadó",
    "monteria": "Montería", "sabaneta": "Sabaneta", "armenia": "Armenia", "soledad": "Soledad",
    "lenguazaque": "Lenguazaque", "candelaria": "Candelaria", "piedecuesta": "Piedecuesta",
    "riofrio": "Riofrío", "palmira": "Palmira", "tenjo": "Tenjo", "jamundi": "Jamundí",
    "el carmen de viboral": "El Carmen de Viboral", "turbaco": "Turbaco", "palermo": "Palermo",
    "cajicá": "Cajicá", "neiva": "Neiva", "valledupar": "Valledupar", "guacari": "Guacarí",
    "envigado": "Envigado",
    "huy": None,
}
clave = df["ciudad"].str.strip().str.lower()
sin_mapear = set(clave.dropna()) - set(CIUDADES)
assert not sin_mapear, sin_mapear

df["ciudad"] = clave.map(CIUDADES).fillna("Sin dato")
df["ciudad"].value_counts()

## 5. Recibir información
Aquí solo hay `Sí` y `No` (no existe la opción "Otros" del 03). Si llegara otra respuesta, queda como `Sin respuesta`.

In [ ]:
recibir = df["recibir_info"].str.strip()
df["recibir_info"] = recibir.where(recibir.isin(["Sí", "No"])).fillna("Sin respuesta")
df["recibir_info"].value_counts()

## 6. Temas sugeridos
Se quitan espacios y se vacían las respuestas que **no proponen un tema**:
- Sin contenido: `.`, `-`, `Lo`.
- "No aplica" escrito de varias formas: `N/a`, `Na`, `N\A`, `N A`, `No aplica`, `Ninguno`.
- Comentarios que no son temas: `Si`, `Excelente`, `Maravilloso!`, `Gracias`.

`IA` se conserva (inteligencia artificial).

In [ ]:
SIN_TEMA = {".", "-", "lo", "si", "no", "n/a", "na", "n\\a", "n a", "no aplica", "ninguno",
            "excelente", "maravilloso!", "gracias"}
temas = df["temas_sugeridos"].str.strip().replace({"Ia": "IA"})
df["temas_sugeridos"] = temas.mask(temas.str.lower().isin(SIN_TEMA))
print("Respuestas con tema:", df["temas_sugeridos"].notna().sum())
df["temas_sugeridos"].dropna().sort_values(key=lambda s: s.str.len()).head(10)

## 7. Anónimas y duplicados
- Las 58 respuestas sin persona **se conservan** (sus calificaciones cuentan para los promedios) y se marcan con `anonima`.
- Duplicados: misma persona, mismo día y mismas 4 calificaciones. Hay 1 (P-02743 respondió dos veces con 6 minutos de diferencia). Se conserva la primera.

In [ ]:
CALIFICACIONES = ["cal_plataforma", "cal_convocatoria", "cal_conferencistas", "cal_temas"]

df["anonima"] = df["id_persona"].isna()
duplicada = df.duplicated(["id_persona", "fecha_evento"] + CALIFICACIONES) & ~df["anonima"]
print("Eliminadas:", df.loc[duplicada, "id_respuesta"].tolist())

df = df[~duplicada].reset_index(drop=True)
assert len(df) == 317, len(df)

## 8. Validaciones y columnas finales
Mismo orden de columnas que el 03, con `cal_plataforma` en el lugar de `cal_locacion` y `modalidad = "Virtual"`.

In [ ]:
assert df[CALIFICACIONES].isin([1, 2, 3, 4, 5]).all().all()
assert df["id_respuesta"].is_unique
assert df["id_empresa"].dropna().str.fullmatch(r"EMP-\d{4}").all()
assert df["id_persona"].dropna().str.fullmatch(r"P-\d{5}").all()

df["cal_promedio"] = df[CALIFICACIONES].mean(axis=1).round(2)
df["modalidad"] = "Virtual"

df = df[["id_respuesta", "fecha_inicio", "fecha_fin", "fecha_evento", "anio", "mes",
         *CALIFICACIONES, "cal_promedio", "temas_sugeridos", "recibir_info",
         "id_persona", "id_empresa", "ciudad", "empresa_en_crm", "metodo_asignacion",
         "anonima", "modalidad"]]
df.dtypes

## 9. Guardar
`utf-8-sig` para que Excel y Looker Studio lean bien las tildes.

In [ ]:
salida = LIMPIOS / "04_eventos_virtuales_limpio.csv"
df.to_csv(salida, index=False, encoding="utf-8-sig")

assert pd.read_csv(salida).shape == (317, 20)
salida.name